In [1]:
pip install pandas yfinance datasets spacy tqdm opendatasets transformers torch

  Using cached pandas-3.0.2-cp311-cp311-win_amd64.whl (9.9 MB)
  Using cached yfinance-1.2.0-py2.py3-none-any.whl (130 kB)
  Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
  Using cached spacy-3.8.14-cp311-cp311-win_amd64.whl (15.4 MB)
  Using cached opendatasets-0.1.22-py3-none-any.whl (15 kB)
  Using cached transformers-5.4.0-py3-none-any.whl (10.1 MB)
  Using cached torch-2.11.0-cp311-cp311-win_amd64.whl (114.5 MB)
  Using cached numpy-2.4.4-cp311-cp311-win_amd64.whl (12.6 MB)
  Using cached requests-2.33.1-py3-none-any.whl (64 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl (16 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl (1.6 MB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl (437 kB)
  Using cached filelock-3.25.2-py3-none-any.whl (26 kB)
  Using cached pyarrow-23.0.1-cp311-cp311-win_amd64.whl (27.5 MB)
  Using cached dill-0.4.1-py3-none-any.whl (120 kB)
  Using cached httpx-0.28.1-

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\Calmen\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\numpy\\lib\\tests\\test_io.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!python -m spacy download en_core_web_md

c:\Users\Calmen\AppData\Local\Programs\Python\Python311\python.exe: No module named spacy


MasterData Building

In [ ]:
import os
import requests
import pandas as pd
import time
from datetime import datetime
from calendar import monthrange
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
BASE_URL = "https://www.alphavantage.co/query"

MY_TICKERS = {
    'CL=F', 'USO', 'XOM', 'CVX', 'BP', 'SHEL', 'COP', 'EOG', 'OXY', 'XLE', 'XOP',
    'HO=F', 'CRAK', 'VLO', 'MPC', 'PSX', 'NG=F', 'UNG', 'LNG', 'TELL', 'EQT', 'RRC'
}

def harvest_chunk(year: int, month: int) -> list:
    """Fetch up to 1000 articles for one month."""
    last_day = monthrange(year, month)[1]
    time_from = f"{year}{month:02d}01T0000"
    time_to = f"{year}{month:02d}{last_day:02d}T2359"

    params = {
        "function": "NEWS_SENTIMENT",
        "topics": "energy",
        "time_from": time_from,
        "time_to": time_to,
        "limit": 1000,
        "sort": "LATEST",
        "apikey": API_KEY,
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        # Alpha Vantage often returns a "Note" when rate limited
        if "Note" in data:
            print(f"Rate limit / API note for {year}-{month:02d}: {data['Note']}")
            return []

        if "Information" in data:
            print(f"API info for {year}-{month:02d}: {data['Information']}")
            return []

        return data.get("feed", [])

    except Exception as e:
        print(f"Error on {year}-{month:02d}: {e}")
        return []

def check_relevance(ticker_list) -> bool:
    if not isinstance(ticker_list, list):
        return False
    found = {item.get("ticker") for item in ticker_list if isinstance(item, dict)}
    return not found.isdisjoint(MY_TICKERS)

if not API_KEY:
    raise ValueError("Missing ALPHAVANTAGE_API_KEY environment variable")

current_year = datetime.now().year
current_month = datetime.now().month

# Example: harvest 2024 through current year
years_to_harvest = [2022,2023]

all_data = []

print(f"Starting harvest for years: {years_to_harvest}")

for year in years_to_harvest:
    for month in range(1, 13):
        if year == current_year and month > current_month:
            continue

        print(f"Pulling {year}-{month:02d}...")
        batch = harvest_chunk(year, month)
        all_data.extend(batch)

        print(f"Collected {len(batch)} articles. Waiting 15s...")
        time.sleep(15)

df_raw = pd.DataFrame(all_data)

if df_raw.empty:
    print("No data found. Check API key, rate limits, or query filters.")
else:
    if "ticker_sentiment" not in df_raw.columns:
        raise KeyError("ticker_sentiment column missing from API response")

    print("Filtering for relevant tickers...")
    df_raw["is_relevant"] = df_raw["ticker_sentiment"].apply(check_relevance)
    master_df = df_raw[df_raw["is_relevant"]].copy()

    if "url" in master_df.columns:
        master_df = master_df.drop_duplicates(subset=["url"])

    if "time_published" in master_df.columns:
        master_df["dt"] = pd.to_datetime(
            master_df["time_published"],
            format="%Y%m%dT%H%M%S",
            errors="coerce"
        )
        master_df = master_df.sort_values("dt", ascending=False)

    filename = f"Energy_BigData_2022_2023.csv"
    master_df.to_csv(filename, index=False)

    print("\nHARVEST COMPLETE")
    print(f"Total raw articles: {len(df_raw)}")
    print(f"Total relevant articles: {len(master_df)}")
    print(f"Saved to: {filename}")

Starting harvest for years: [2024, 2025]
Pulling 2024-01...
Collected 1000 articles. Waiting 15s...
Pulling 2024-02...
Collected 1000 articles. Waiting 15s...
Pulling 2024-03...
Collected 1000 articles. Waiting 15s...
Pulling 2024-04...
Collected 1000 articles. Waiting 15s...
Pulling 2024-05...
Collected 1000 articles. Waiting 15s...
Pulling 2024-06...
Collected 1000 articles. Waiting 15s...
Pulling 2024-07...
Collected 1000 articles. Waiting 15s...
Pulling 2024-08...
Collected 1000 articles. Waiting 15s...
Pulling 2024-09...
Collected 1000 articles. Waiting 15s...
Pulling 2024-10...
Collected 1000 articles. Waiting 15s...
Pulling 2024-11...
Collected 1000 articles. Waiting 15s...
Pulling 2024-12...
Collected 1000 articles. Waiting 15s...
Pulling 2025-01...
Collected 1000 articles. Waiting 15s...
Pulling 2025-02...
Collected 1000 articles. Waiting 15s...
Pulling 2025-03...
Collected 1000 articles. Waiting 15s...
Pulling 2025-04...
Collected 1000 articles. Waiting 15s...
Pulling 2025-05

In [ ]:
import pandas as pd
df=pd.read_csv("Energy_BigData_2022_2023.csv")

In [2]:
df.head()

,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,is_relevant,dt
0,ExxonMobil announces third-quarter 2025 results,https://corporate.exxonmobil.com/news/news-rel...,20251031T185433,[],Exxon Mobil Corporation announced strong third...,NaN,Exxon Mobil Corporation | ExxonMobil,General,Exxon Mobil Corporation | ExxonMobil,"[{'topic': 'earnings', 'relevance_score': '1.0...",0.417328,Bullish,"[{'ticker': 'XOM', 'relevance_score': '1.00000...",True,2025-10-31 18:54:33
1,Palmetto Grain Brokerage -,https://www.palmettograin.com/news/story/35827...,20251031T180110,['Neha Panjwani'],This article from Barchart discusses whether W...,https://media.barchart.com/contributors-admin/...,Palmetto Grain Brokerage,General,Palmetto Grain Brokerage,"[{'topic': 'financial_markets', 'relevance_sco...",-0.028607,Neutral,"[{'ticker': 'DVN', 'relevance_score': '0.93333...",True,2025-10-31 18:01:10
2,EQT Corp. stock outperforms competitors on str...,https://www.marketwatch.com/data-news/eqt-corp...,20251031T165400,['MarketWatch Automation'],EQT Corp. (EQT) saw its stock advance by 2.15%...,NaN,MarketWatch,General,MarketWatch,"[{'topic': 'financial_markets', 'relevance_sco...",0.413139,Bullish,"[{'ticker': 'EQT', 'relevance_score': '1.00000...",True,2025-10-31 16:54:00
3,EQT Corp. stock outperforms competitors on str...,https://www.marketwatch.com/data-news/eqt-corp...,20251031T165400,['MarketWatch Automation'],"EQT Corp. stock saw a gain of 2.15% on Friday,...",NaN,MarketWatch,General,MarketWatch,"[{'topic': 'financial_markets', 'relevance_sco...",0.461938,Bullish,"[{'ticker': 'EQT', 'relevance_score': '1.00000...",True,2025-10-31 16:54:00
4,EQT Corp. stock outperforms competitors on str...,https://www.marketwatch.com/data-news/eqt-corp...,20251031T165400,['MarketWatch Automation'],EQT Corp. (EQT) stock advanced 2.15% to $53.58...,NaN,MarketWatch,General,MarketWatch,"[{'topic': 'financial_markets', 'relevance_sco...",0.517154,Bullish,"[{'ticker': 'EQT', 'relevance_score': '1.00000...",True,2025-10-31 16:54:00


In [3]:
import ast
import pandas as pd

SC_MAP = {
    # Upstream
    "CL=F": ("crude oil", "Upstream"),
    "USO": ("crude oil", "Upstream"),
    "XOM": ("crude oil", "Upstream"),
    "CVX": ("crude oil", "Upstream"),
    "COP": ("crude oil", "Upstream"),
    "EOG": ("crude oil", "Upstream"),
    "OXY": ("crude oil", "Upstream"),
    "BP": ("crude oil", "Integrated"),
    "SHEL": ("crude oil", "Integrated"),

    # LNG / natural gas
    "EQT": ("LNG", "Upstream"),
    "RRC": ("LNG", "Upstream"),
    "LNG": ("LNG", "Midstream"),
    "TELL": ("LNG", "Midstream"),
    "UNG": ("LNG", "Midstream"),
    "NG=F": ("LNG", "Midstream"),

    # Downstream / refined products
    "VLO": ("diesel", "Downstream"),
    "MPC": ("diesel", "Downstream"),
    "PSX": ("diesel", "Downstream"),
    "HO=F": ("diesel", "Downstream"),
    "CRAK": ("diesel", "Downstream"),
}


def safe_parse(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []


def extract_commodities(ticker_sentiment, sc_map=SC_MAP):
    ticker_list = safe_parse(ticker_sentiment)
    commodities = []

    for item in ticker_list:
        if isinstance(item, dict):
            ticker = item.get("ticker")
            if ticker in sc_map:
                commodity = sc_map[ticker][0]
                if commodity not in commodities:
                    commodities.append(commodity)

    return ",".join(commodities) if commodities else None


def parse_energy_to_merged_format(df_energy):
    df = df_energy.copy()

    df["commodity"] = df["ticker_sentiment"].apply(extract_commodities)
    df["date"] = pd.to_datetime(df["dt"], errors="coerce").dt.strftime("%Y-%m-%d")
    df["description"] = df["summary"]

    result = df[["date", "title", "description", "commodity"]].copy()

    result = result.dropna(subset=["date", "title", "description"])
    result = result.reset_index(drop=True)

    return result
df_parsed = parse_energy_to_merged_format(df)

In [8]:
import pandas as pd

# Load original merged_df if needed
merged_df = pd.read_csv("merged_df.csv")

merged_base = merged_df[["date", "title", "description", "commodity"]].copy()
parsed_base = df_parsed[["date", "title", "description", "commodity"]].copy()
final_df = pd.concat([merged_base, parsed_base], ignore_index=True)
final_df = final_df.drop_duplicates(subset=["date", "title", "description", "commodity"])
final_df["date"] = pd.to_datetime(final_df["date"], errors="coerce")
final_df = final_df.sort_values("date").reset_index(drop=True)
final_df["date"] = final_df["date"].dt.strftime("%Y-%m-%d")
final_df.head()


,date,title,description,commodity
0,2022-01-01,Some Of The Biggest Lies Being Spread About Th...,The events from last Jan. 6 in Washington unfo...,NaN
1,2022-01-01,Austrian Holocaust Survivor 'Mrs. Gertrude' Di...,Gertrude Pressburger became famous during Aust...,NaN
2,2022-01-01,Rep. Alexandria Ocasio-Cortez Rips 'Creepy Wei...,The congresswoman fired back at Republican Ste...,NaN
3,2022-01-02,Chilling Trump Letter Calling For 'Seizure' Of...,The letter was created a day before Trump disc...,NaN
4,2022-01-02,Marjorie Taylor Greene's Personal Twitter Acco...,The Georgia Republican was given the boot afte...,NaN


In [12]:
final_df=final_df[final_df["commodity"].notna() & (final_df["commodity"] != "")]

In [14]:
final_df['date'] = pd.to_datetime(final_df['date'], errors='coerce')
print(final_df['date'].min(), final_df['date'].max())
final_df['year'] = final_df['date'].dt.year
final_df['month'] = final_df['date'].dt.month
final_df['year_month'] = final_df['date'].dt.to_period('M').astype(str)
print(sorted(final_df['year_month'].dropna().unique()))

2022-01-03 00:00:00 2026-03-31 00:00:00
['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2026-03']


Labelling

Using LLMs to label domain and extract risk keywords

In [15]:
import json
import time
import pandas as pd

SYSTEM_INSTRUCTION_EXPAND = """
You are an expert Energy Market & Supply Chain Analyst. 
Your task is to analyze news title and description to categorize energy news for a high-precision Risk Management RAG system.

Assign each item:
1. news_category
2. risk_category
3. risk_severity
4. market_sentiment

--- PREDEFINED NEWS CATEGORIES ---
1. Geopolitics & policy
2. Supply (production / upstream)
3. Refining (downstream)
4. Inventory & storage
5. Demand / macro activity
6. LNG & natural gas
7. Weather
8. Shipping & logistics
9. Financial flows / positioning
10. Currency & interest rates
11. Accidents / disruptions
12. Energy transition / structural

--- PREDEFINED RISK CATEGORIES ---
- Supply Chain Blockage
- Production Shortfall
- Refining Outage
- Geopolitical Conflict
- Macro-Economic Cooling
- Inventory Shock
- Infrastructure Damage
- Weather Extremity
- Regulatory Constraint
- No Significant Risk

--- PREDEFINED RISK SEVERITY ---
- Low
- Medium
- High

--- PREDEFINED MARKET SENTIMENT ---
- Bullish
- Bearish
- Neutral

Rules:
- Choose exactly one news_category from the predefined list.
- Choose exactly one risk_category from the predefined list.
- Choose exactly one risk_severity from: Low, Medium, High.
- Choose exactly one market_sentiment from: Bullish, Bearish, Neutral.
- Base the label on the title + description only.
- If the article is mainly about company earnings, stock movement, analyst ratings, or financing, prefer:
  news_category = Financial flows / positioning
- If the article is mainly about LNG, gas supply, pipelines, or gas exports/imports, prefer:
  news_category = LNG & natural gas
- If no material risk is present, use:
  risk_category = No Significant Risk
  risk_severity = Low
- market_sentiment should reflect likely commodity-market impact, not just company-level stock tone.

--- OUTPUT FORMAT ---
Return a JSON list.
Each object must contain exactly:
{
  "id": <input id>,
  "news_category": "<one predefined news category>",
  "risk_category": "<one predefined risk category>",
  "risk_severity": "<Low|Medium|High>",
  "market_sentiment": "<Bullish|Bearish|Neutral>"
}
"""


In [16]:
def build_batch_prompt(batch_df):
    prompt_input = ""
    for idx, row in batch_df.iterrows():
        title = str(row.get("title", "")).strip()
        description = str(row.get("description", "")).strip()
        commodities = str(row.get("relevant_commodities", "")).strip()

        prompt_input += (
            f"ID: {idx} | "
            f"Title: {title} | "
            f"Description: {description} | "
            f"Relevant Commodities: {commodities}\n"
        )

    return SYSTEM_INSTRUCTION_EXPAND + "\n\nAnalyze these items:\n" + prompt_input


In [17]:
def expand_categories_with_llm(
    df,
    client,
    model="gpt-4o-mini",
    batch_size=20,
    sleep_seconds=0.5
):
    working_df = df.copy()

    if "commodity" in working_df.columns:
        working_df = working_df.rename(columns={"commodity": "relevant_commodities"})

    required_cols = ["date", "title", "description", "relevant_commodities"]
    missing = [c for c in required_cols if c not in working_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    all_results = []

    for start in range(0, len(working_df), batch_size):
        batch = working_df.iloc[start:start + batch_size]
        prompt = build_batch_prompt(batch)

        response = client.responses.create(
            model=model,
            input=prompt,
        )

        text = response.output_text.strip()
        text = text.replace("```json", "").replace("```", "").strip()

        try:
            batch_results = json.loads(text)
        except json.JSONDecodeError as e:
            print(f"JSON parse error for batch starting at row {start}: {e}")
            print(text[:1000])
            continue

        all_results.extend(batch_results)
        print(f"Processed rows {start} to {start + len(batch) - 1}")
        time.sleep(sleep_seconds)

    labels_df = pd.DataFrame(all_results)

    if labels_df.empty:
        raise ValueError("No labels returned from LLM.")

    if "id" not in labels_df.columns:
        raise ValueError("LLM output must contain 'id'.")

    labels_df["id"] = labels_df["id"].astype(int)

    merged = (
        working_df.reset_index()
        .merge(labels_df, left_on="index", right_on="id", how="left")
        .drop(columns=["index", "id"])
    )

    final_cols = [
        "date",
        "title",
        "description",
        "relevant_commodities",
        "news_category",
        "risk_category",
        "risk_severity",
        "market_sentiment",
    ]

    return merged[final_cols]


In [ ]:
# pip install openai

     ---------------------------------------- 1.1/1.1 MB 14.6 MB/s eta 0:00:00
     ---------------------------------------- 204.6/204.6 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from openai import OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

expanded_df = expand_categories_with_llm(
    df=final_df,
    client=client,
    model="gpt-4o-mini",
    batch_size=20
)

expanded_df.head()


Processed rows 0 to 19
Processed rows 20 to 39
Processed rows 40 to 59
Processed rows 60 to 79
Processed rows 80 to 99
Processed rows 100 to 119
Processed rows 120 to 139
Processed rows 140 to 159
Processed rows 160 to 179
Processed rows 180 to 199
Processed rows 200 to 219
Processed rows 220 to 239
Processed rows 240 to 259
Processed rows 260 to 279
Processed rows 280 to 299
Processed rows 300 to 319
Processed rows 320 to 339
Processed rows 340 to 359
Processed rows 360 to 379
Processed rows 380 to 399
Processed rows 400 to 419
Processed rows 420 to 439
Processed rows 440 to 459
Processed rows 460 to 479
Processed rows 480 to 483


,date,title,description,relevant_commodities,news_category,risk_category,risk_severity,market_sentiment
0,2022-01-03,"After Maharashtra, distributors might boycott ...",FMCG product distributors are planning to boyc...,crude oil,Supply (production / upstream),No Significant Risk,Low,Neutral
1,2022-01-03,Colgate items to go off shelves in Maharashtra,FMCG distributors in Maharashtra have announce...,crude oil,Supply (production / upstream),No Significant Risk,Low,Neutral
2,2022-01-04,Colgate in talks with India sales agents after...,Colgate-Palmolive India is in talks with its s...,crude oil,Supply (production / upstream),No Significant Risk,Low,Neutral
3,2022-01-04,Colgate in talks with India sales agents after...,Colgate-Palmolive India is in discussions with...,crude oil,Supply (production / upstream),No Significant Risk,Low,Neutral
4,2022-01-04,US becomes world’s top LNG exporter for first ...,The United States became the world's top expor...,LNG,LNG & natural gas,No Significant Risk,Low,Bullish


In [25]:
def show_counts_only(df):
    cols = ["news_category", "risk_category", "risk_severity", "market_sentiment"]

    for col in cols:
        print(f"\n=== {col} ===")
        print(df[col].fillna("Missing").value_counts())

    print("\n=== relevant_commodities ===")
    commodity_counts = (
        df["relevant_commodities"]
        .fillna("Missing")
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
        .value_counts()
    )
    print(commodity_counts)
show_counts_only(expanded_df)





=== news_category ===
news_category
Financial flows / positioning     150
LNG & natural gas                  98
Geopolitics & policy               81
Supply (production / upstream)     71
Energy transition / structural     28
Refining (downstream)              24
Demand / macro activity            17
Accidents / disruptions             5
Shipping & logistics                4
No Significant Risk                 2
Inventory & storage                 2
Weather                             1
Missing                             1
Name: count, dtype: int64

=== risk_category ===
risk_category
No Significant Risk               335
Geopolitical Conflict              41
Production Shortfall               33
Regulatory Constraint              24
Supply Chain Blockage              18
Infrastructure Damage              12
Refining Outage                     6
Macro-Economic Cooling              5
Inventory Shock                     4
Supply (production / upstream)      2
Demand / macro activity   

In [26]:
expanded_df.to_csv("final_expanded_df.csv", index=False)